# Building Intelligent Agents with LlamaIndex

LlamaIndex enables **agents** that interact with external tools, retrieve structured data, and execute tasks beyond simple LLM responses.

### This notebook covers:
- Setting up agents with tools (e.g., APIs, query engines)  
- Using ReAct and OpenAI agents for reasoning  
- Configuring planning agents for complex tasks  
- Building custom agents for workflows  

## Understanding Agents in LlamaIndex  

Agents process queries through:  
1. **Breaking down complex queries** into sub-questions.  
2. **Selecting tools** (e.g., APIs, databases, function calls).  
3. **Executing multi-step workflows** with structured plans.  
4. **Using memory** to retain past interactions for improved responses.  

Agents can handle both single-response and multi-step queries, adapting based on prior exchanges.

## Setting Up: Agent and Tool Use  

Let's restore resources from previous notebooks before configuring agents.

In [ ]:
%load_ext ipython_async

In [ ]:
%%asynccmd
mkdir .\vectorstore\postgres &
initdb -D .\vectorstore\postgres --encoding=UTF8 --locale=en_US.UTF-8 &
pg_ctl -D .\vectorstore\postgres -l logfile start

In [ ]:
import os

In [ ]:
db_name = "vector_db"
user = "acisse"

In [ ]:
from llama_index.vector_stores.postgres import PGVectorStore

vector_store = PGVectorStore.from_params(
    database=db_name,
    host="localhost",
    password=os.getenv("PSQL_PASSWORD"),
    port=5432,
    user=user,
    table_name="physics_docs",
    embed_dim=384,
    hybrid_search=True
)

In [ ]:
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain.embeddings import HuggingFaceEmbeddings

# Load MiniLM embedding model (fast & efficient)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Wrap the model for LlamaIndex
embed_model = LangchainEmbedding(embedding_model)

Finally, we can restore the index:

In [ ]:
from pathlib import Path
from llama_index.core import StorageContext, load_index_from_storage

# Load the stored index
index_save_path = Path("assets", "llama_vector_search", "physics_documents_index")
storage_context = StorageContext.from_defaults(persist_dir=index_save_path, vector_store=vector_store)
index = load_index_from_storage(storage_context, embed_model=embed_model)

And load our secrets into the environment:

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Query Planning Engine

LlamaIndex agents refine and restructure user queries for better retrieval by:  
- Breaking broad queries into **sub-questions**  
- Reformulating ambiguous queries into **retrieval-friendly formats**  
- Synthesizing multiple retrieved results into a **coherent response**  

### Question Generation  

LlamaIndex’s **QuestionGenerator** module extracts key concepts to create sub-questions, enhancing **retrieval evaluation**, **dataset expansion**, and **self-supervised learning**.


In [ ]:
from llama_index.core.question_gen.prompts import DEFAULT_SUB_QUESTION_PROMPT_TMPL, PromptTemplate
from llama_index.core.question_gen.output_parser import SubQuestionOutputParser
from utils import display_citation
display_citation(DEFAULT_SUB_QUESTION_PROMPT_TMPL)

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core.tools import ToolMetadata
llm_openai = OpenAI(model="gpt-4o")
tool_metadata = ToolMetadata(
        name="physics_documents",
        description="Contains informations on core topics of physics, as well as biographies of important physisists",
    )

In [ ]:
from llama_index.core.question_gen.llm_generators import LLMQuestionGenerator
from llama_index.core.schema import QueryBundle
LLMQuestionGenerator
# Initialize the question generator
question_generator = LLMQuestionGenerator(
    llm=llm_openai,
    prompt=PromptTemplate(DEFAULT_SUB_QUESTION_PROMPT_TMPL, output_parser=SubQuestionOutputParser())
)

query = QueryBundle("How did the research at los alamos and bell labs overlap?")

# Generate questions based on the indexed content
questions = question_generator.generate(
    tools=[tool_metadata], 
    query=query
)
for question in questions:
    print(f"Uses: {question.tool_name}")
    display_citation(question.sub_question)

Note, that an openai equivalent exists:

In [ ]:
from llama_index.question_gen.openai import OpenAIQuestionGenerator

llm = OpenAI()
question_gen = OpenAIQuestionGenerator.from_defaults(llm=llm_openai)

questions = question_gen.generate([tool_metadata], query)
for question in questions:
    print(f"Uses: {question.tool_name}")
    display_citation(question.sub_question)

### Pre-Agentic Subquery Engine

LlamaIndex provides a **SubQuestionQueryEngine**, which automatically decomposes a complex query into multiple sub-queries and retrieves answers for each before synthesizing a final response. 

In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool
from llama_index.core.response_synthesizers import get_response_synthesizer


# Create a query engine from the index
query_engine = index.as_query_engine()

# Wrap the query engine in a QueryEngineTool
query_engine_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=tool_metadata
)

# Create a response synthesizer (optional, improves result formatting)
response_synthesizer = get_response_synthesizer(response_mode="refine")

# Initialize the SubQuestionQueryEngine using from_defaults
sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[query_engine_tool],     # Must be a list
    llm=llm_openai,                             # Optional: specify an LLM
    response_synthesizer=response_synthesizer,  # Optional
    verbose=True,                               # Enable detailed output
    use_async=False
)

# Ask a complex question
response = sub_question_engine.query("How did the research at los alamos and bell labs overlap?")
display_citation(response.response)

### Optimizing Retrieval with Query HyDE  

**Query transformations** refine input queries by:  
- Expanding context  
- Rephrasing for relevance  
- Generating variations for broader search  

**HyDE** improves retrieval by generating a hypothetical response, embedding it instead of the raw query, and enhancing semantic similarity with indexed documents.

In [ ]:
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.core.indices.query.query_transform import HyDEQueryTransform

# Apply a Hypothetical Document Expansion (HyDE) transformation
hyde_transform = HyDEQueryTransform()
display_citation(hyde_transform.get_prompts()['hyde_prompt'].template)


Using this prompt, we obtain the following output:

In [ ]:
query = QueryBundle("How did the research at los alamos and bell labs overlap?")
queries = hyde_transform.run(query_bundle_or_str=query)
for query in queries.custom_embedding_strs:
    display_citation(query)

In [ ]:
# Wrap the base query engine with the transformation
query_engine = TransformQueryEngine(
    query_engine=index.as_query_engine(),
    query_transform=hyde_transform
)

# Ask a query with automatic expansion
response = query_engine.query("How did the research at los alamos and bell labs overlap?")
display_citation(response.response)

In [ ]:
from llama_index.core import PromptTemplate
from llama_index.llms.openai import OpenAI

query_gen_str = """\
You are a helpful assistant that generates multiple search queries based on a \
single input query. Generate {num_queries} search queries, one on each line, \
related to the following input query:
Query: {query}
Queries:
"""
query_gen_prompt = PromptTemplate(query_gen_str)

llm = OpenAI(model="gpt-3.5-turbo")


def generate_queries(query: str, llm, num_queries: int = 4):
    response = llm.predict(
        query_gen_prompt, num_queries=num_queries, query=query
    )
    # assume LLM proper put each query on a newline
    queries = response.split("\n")
    queries_str = "\n".join(queries)
    print(f"Generated queries: ")
    display_citation(queries_str)
    return queries

queries = generate_queries("How did the research at los alamos and bell labs overlap?", llm)

## Building Agents for Adaptive Retrieval  

Traditional retrieval executes a single search, but **agents** refine queries iteratively to improve accuracy. They recognize incomplete results, generate follow-up searches, and adjust their approach dynamically.

### Agent Architectures:
- **Structured Planner Agents**: Break queries into multiple retrieval tasks.  
- **Introspective Agents**: Self-evaluate responses and issue follow-up queries.  

These agents enable **multi-step reasoning**, ensuring no relevant information is missed.

### Structured Planner Agents: Query Decomposition  

Structured planner agents divide complex queries into **sequential retrieval steps**, ensuring thorough search coverage.

Useful when:
- A query requires multiple sources.  
- A question has multiple subtopics.  
- The best retrieval strategy isn’t clear.  

They follow two steps:
1. **Generate a query plan**, defining the necessary retrieval steps.  
2. **Execute the plan**, retrieving and synthesizing information.  

This method ensures **comprehensive, well-organized retrieval**, minimizing missing data.

In [ ]:
from llama_index.core.agent import StructuredPlannerAgent, FunctionCallingAgentWorker
from llama_index.core.tools import QueryEngineTool
import nest_asyncio
nest_asyncio.apply()

# Create a query engine from our index
query_engine = index.as_query_engine()

# Wrap the query engine in a QueryEngineTool
query_engine_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="vector_index",
        description="A tool for retrieving information from a physics document index."
    )
)

# Create a function-calling worker for executing retrieval tasks
worker = FunctionCallingAgentWorker.from_tools(
    [query_engine_tool], 
    llm=llm_openai, 
    verbose=True,
    allow_parallel_tool_calls=False
)

# Initialize the structured planner agent with the worker
structured_planner = StructuredPlannerAgent(
    worker, 
    tools=[query_engine_tool], 
    verbose=True
)

# Test with a complex question
response = structured_planner.chat("How did research at Los Alamos and Bell Labs overlap?")
print(response)


In [ ]:
for value in structured_planner.state.plan_dict.values():
    for task in value.sub_tasks:
        print(f"Name: {task.name}")
        print(f"Input: {task.input}")
        print(f"Expected output:")
        display_citation(task.expected_output)

You can modify the agent using a lower level apprach. Visit here to learn more on how to use the [low level api](https://docs.llamaindex.ai/en/stable/examples/agent/structured_planner/) for the agent.

https://docs.llamaindex.ai/en/stable/examples/agent/structured_planner/
https://docs.llamaindex.ai/en/stable/api_reference/agent/introspective/